# 08.08 - Self-supervised vision fundamentals

**Notebook type:** Solution notebook with completed exercises, smoke checks, and test cases.

**Daily output:** Two-view pipeline and contrastive pretraining step.

Create two stochastic views of each unlabeled image, encode them, and optimize a compact symmetric contrastive objective with in-batch negatives.

## Core Ideas

Self-supervised contrastive learning creates supervision from identity: two views of the same observation are positives, while other observations in the batch are negatives. Augmentations must preserve semantic identity. Embeddings are normalized before cosine similarity, and temperature controls how sharply similarities affect the loss.

In [ ]:
import numpy as np
import torch
from torch import nn

SEED = 8
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Prepared Unlabeled Images

Twenty-four tiny images contain structured bars but no labels. Data generation is complete setup code.

In [ ]:
unlabeled_images = torch.zeros((24, 1, 8, 8), dtype=torch.float32)
for index in range(24):
    if index % 2:
        unlabeled_images[index, 0, :, 2 + index % 4] = 1.0
    else:
        unlabeled_images[index, 0, 2 + index % 4, :] = 1.0
print("unlabeled fixture:", unlabeled_images.shape, unlabeled_images.dtype)

## Exercise 08-A: Create paired stochastic views

Apply independent bounded noise to the same image batch while preserving shape and value range.

**Return structure — `make_two_views`:** A tuple `(view_a, view_b)` of CPU float32 tensors matching input shape `[N,C,H,W]`, with values clipped to `[0,1]` and non-identical views.

In [ ]:
def make_two_views(images, noise_std=0.08, seed=SEED):
    images = images.detach().cpu().to(torch.float32)
    first_generator = torch.Generator().manual_seed(int(seed))
    second_generator = torch.Generator().manual_seed(int(seed) + 1)
    first_noise = torch.randn(images.shape, generator=first_generator) * noise_std
    second_noise = torch.randn(images.shape, generator=second_generator) * noise_std
    return (images + first_noise).clamp(0, 1), (images + second_noise).clamp(0, 1)


# Smoke check: create both views for every observation.
view_a, view_b = make_two_views(unlabeled_images)
print(view_a.shape, float((view_a - view_b).abs().mean()))

## Exercise 08-B: Build encoder and projection head

Keep the encoder representation separate from the projection used by the contrastive loss.

**Return structure — `build_ssl_modules`:** A dictionary with `encoder` and `projector` modules on `device`. Encoder maps `[N,1,8,8]→[N,16]`; projector maps `[N,16]→[N,8]`.

In [ ]:
def build_ssl_modules(device=DEVICE):
    encoder = nn.Sequential(nn.Flatten(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 16)).to(device)
    projector = nn.Sequential(nn.ReLU(), nn.Linear(16, 8)).to(device)
    return {"encoder": encoder, "projector": projector}


# Smoke check: verify representation and projection shapes.
ssl_modules = build_ssl_modules()
smoke_features = ssl_modules["encoder"](view_a[:4].to(DEVICE))
print(smoke_features.shape, ssl_modules["projector"](smoke_features).shape)

## Exercise 08-C: Calculate symmetric contrastive loss

Use the diagonal as positive matches and all other cross-view observations as in-batch negatives.

**Return structure — `symmetric_contrastive_loss`:** A scalar tensor on the embeddings device. Inputs share shape `[N,D]`; normalized cross-view logits have shape `[N,N]`.

In [ ]:
def symmetric_contrastive_loss(projection_a, projection_b, temperature=0.2):
    first = nn.functional.normalize(projection_a, dim=1)
    second = nn.functional.normalize(projection_b, dim=1)
    logits = first @ second.T / float(temperature)
    targets = torch.arange(len(first), device=first.device)
    return 0.5 * (nn.functional.cross_entropy(logits, targets) + nn.functional.cross_entropy(logits.T, targets))


# Smoke check: calculate an untrained contrastive loss.
projection_a = ssl_modules["projector"](ssl_modules["encoder"](view_a.to(DEVICE)))
projection_b = ssl_modules["projector"](ssl_modules["encoder"](view_b.to(DEVICE)))
contrastive_loss = symmetric_contrastive_loss(projection_a, projection_b)
print("loss:", float(contrastive_loss.detach().cpu()))

## Exercise 08-D: Run one complete SSL update

Update encoder and projector together on the complete unlabeled batch.

**Return structure — `ssl_update`:** A dictionary with Python floats `loss`, `positive_similarity_before`, and `positive_similarity_after`; the supplied modules are updated once.

In [ ]:
def ssl_update(modules, first_view, second_view, device=DEVICE):
    encoder, projector = modules["encoder"], modules["projector"]
    optimizer = torch.optim.Adam(list(encoder.parameters()) + list(projector.parameters()), lr=0.01)
    first_view, second_view = first_view.to(device), second_view.to(device)
    with torch.no_grad():
        before_a = nn.functional.normalize(projector(encoder(first_view)), dim=1)
        before_b = nn.functional.normalize(projector(encoder(second_view)), dim=1)
        before = float((before_a * before_b).sum(dim=1).mean().cpu())
    optimizer.zero_grad()
    projection_a, projection_b = projector(encoder(first_view)), projector(encoder(second_view))
    loss = symmetric_contrastive_loss(projection_a, projection_b)
    loss.backward(); optimizer.step()
    with torch.no_grad():
        after_a = nn.functional.normalize(projector(encoder(first_view)), dim=1)
        after_b = nn.functional.normalize(projector(encoder(second_view)), dim=1)
        after = float((after_a * after_b).sum(dim=1).mean().cpu())
    return {"loss": float(loss.detach().cpu()), "positive_similarity_before": before, "positive_similarity_after": after}


# Smoke check and full unlabeled-data evidence.
ssl_record = ssl_update(ssl_modules, view_a, view_b)
print("SSL update:", ssl_record)

## Test Cases

**Return structure — `run_day08_tests`:** Returns `None`; assertions and `Day 08 tests passed` communicate success.

In [ ]:
def run_day08_tests():
    assert view_a.shape == view_b.shape == unlabeled_images.shape
    assert view_a.dtype == torch.float32 and not torch.equal(view_a, view_b)
    assert set(ssl_modules) == {"encoder", "projector"}
    assert smoke_features.shape == (4, 16)
    assert contrastive_loss.ndim == 0 and float(contrastive_loss) > 0
    assert set(ssl_record) == {"loss", "positive_similarity_before", "positive_similarity_after"}
    print("Day 08 tests passed")


run_day08_tests()

## Day 08 Checklist

- [ ] Create two identity-preserving views.
- [ ] Separate encoder features from projection outputs.
- [ ] Normalize embeddings before similarity.
- [ ] Use the complete unlabeled batch for the structural update.
- [ ] Run the test cases.